# Credit Card Default (PATE-GAN) 

Author: Ilse Harmers \
Last modified: February 17, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from snsynth import Synthesizer
import snsynth.transform as tf

In [ ]:
# Importing train set.
credit_train = pd.read_csv("./train-test-datasets/credit-card-default/credit_train.csv")
print(credit_train.columns.to_list())
credit_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.ChainTransformer([
        tf.LogTransformer(),
        tf.MinMaxTransformer(lower = np.log(credit_train["LIMIT_BAL"].min()), 
                             upper = np.log(credit_train["LIMIT_BAL"].max()),   # 1 id with LIMIT_BAL = 1000000
                             negative = False) # LIMIT_BAL; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # SEX
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # EDUCATION
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # MARRIAGE
    tf.MinMaxTransformer(lower = credit_train["AGE"].min(), 
                         upper = credit_train["AGE"].max(),   # 1 id with AGE = 79
                         negative = False), # AGE; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_0
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_2
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_3
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_4
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_5
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_6
    ### Change the lower limits to rounded up (to thousands?) numbers.
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-166000.0) + 1)), 
                             upper = np.log(965000.0 + 1), negative = False) # BILL_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-70000.0) + 1)), 
                             upper = np.log(984000.0 + 1), negative = False) # BILL_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-157000.0) + 1)), 
                             upper = np.log(1664000.0 + 1), negative = False) # BILL_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-170000.0) + 1)), 
                             upper = np.log(892000.0 + 1), negative = False) # BILL_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-81000.0) + 1)), 
                             upper = np.log(927000.0 + 1), negative = False) # BILL_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-340000.0) + 1)), 
                             upper = np.log(962000.0 + 1), negative = False) # BILL_AMT6; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(874000.0 + 1), 
                             negative = False) # PAY_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(1684000.0 + 1), 
                             negative = False) # PAY_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(896000.0 + 1), 
                             negative = False) # PAY_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(621000.0 + 1), 
                             negative = False) # PAY_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(427000.0 + 1), 
                             negative = False) # PAY_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(529000.0 + 1), 
                             negative = False) # PAY_AMT6; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # DEFAULT
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 2 * 10^4 rows, then delta = 10^(-5).
delta = 10**np.floor(np.log10(1/credit_train.shape[0]))
print(delta)

In [ ]:
# Synthesizing the dataset with PATE-GAN. 
synth = Synthesizer.create('pategan', epsilon = 1.0, delta = delta, plot_losses = True)
synth.fit(credit_train, transformer = tt, preprocessor_eps = 0.0)

In [ ]:
# Generating a synthetic dataset with the same amount of rows as the original training dataset.
sample = synth.sample(credit_train.shape[0])

In [ ]:
sample.describe()

In [ ]:
# Saving the synthetic dataset as a csv file.
sample.to_csv("./synthetic-datasets_PATE-GAN/credit-card-default/epsi_1/run5/sample3.csv", index = False)